<h1><center>Recommender Systems YSDA Course!</center></h1>
<h1><center>Семинар №2</center></h1>

<center><img src="logo.jpg" width="500" /></center>

**В этом семинаре мы:**
- Познакомимся с датасетом YAMBDA
- Ссылка на оригинальный датасет: https://huggingface.co/datasets/yandex/yambda
- Посмотрим на контест курса: https://www.kaggle.com/competitions/ysda-rec-sys-2026
- Напишем бейзлайн
- Обучим более сложные модели (CatBoost)
- Напишем несколько новых метрик оценки качества ранжирования

**Баллы за пороги:**
- 5 баллов за пробитие 0.06
- 10 баллов за ...
- 15 баллов за ...
- Топ 3 - дополнительные 10 баллов
- Топ 10 - дополнительные 5 баллов

In [1]:
import numpy as np
import polars as pl
import seaborn as sns
import matplotlib.pyplot as plt

from catboost import CatBoostClassifier, Pool

# 🗄 Датасет:

In [2]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

file_path = "likes.parquet"

data = (
    kagglehub.dataset_load(
        KaggleDatasetAdapter.POLARS,
        "thekabeton/ysda-recsys-2026-yambda-dataset/versions/3",
        file_path,
    )
    .collect()
    .sample(1_000_000)
)  # Ограничение для семинара, лучше использовать все данные


In [3]:
item_ids = data["item_id"].unique()

In [4]:
file_path = "test_users.csv"

test_users = kagglehub.dataset_load(
    KaggleDatasetAdapter.POLARS,
    "thekabeton/ysda-recsys-2026-yambda-dataset/versions/3",
    file_path,
).collect()


In [5]:
file_path = "artist_item_mapping_small.parquet"

artists = kagglehub.dataset_load(
    KaggleDatasetAdapter.POLARS,
    "thekabeton/ysda-recsys-2026-yambda-dataset/versions/3",
    file_path,
).collect()


# 🦾 CatBoost

<center><img src="Timesplit1.svg" width="1100" /></center>


Давайте соберём какие-то фичи из данных и обучим на них градиентный бустинг. Нужно не забывать про временные лики. Нельзя давать модели видеть данные из будущего, поэтому фичи для каждого семпла должны быть посчитаны на данных из прошлого. В простейшей схеме предлагается разделить размеченые данные на 3 части:
- Вторая часть - train
- Третья часть - validation
- Первую часть используем для расчёта статистик для трейна
- Для валидации считаем статистики используя первую и вторую части вместе

#### Делим data на 3 части:

In [6]:
SPLIT_1 = 100
SPLIT_2 = 200
SPLIT_3 = 203

SECONDS_IN_DAY = 60 * 60 * 24


def make_parts(
    data, day_split_1: int = 100, day_split_2: int = 200, day_split_3: int = 300
):
    data_part1 = data.filter(pl.col("timestamp") < SECONDS_IN_DAY * day_split_1)
    data_part2 = data.filter(
        (pl.col("timestamp") >= SECONDS_IN_DAY * day_split_1)
        & (pl.col("timestamp") < SECONDS_IN_DAY * day_split_2)
    )
    data_part3 = data.filter(
        (pl.col("timestamp") >= SECONDS_IN_DAY * day_split_2)
        & (pl.col("timestamp") < SECONDS_IN_DAY * day_split_3)
    )
    return data_part1, data_part2, data_part3

In [7]:
data_len_div3 = int(len(data) / 3)

data = data.sort("timestamp")

data_part1, data_part2, data_part3 = make_parts(
    data, day_split_1=SPLIT_1, day_split_2=SPLIT_2, day_split_3=SPLIT_3
)

#### Набираем негативы:

In [8]:
def add_popular_random_negatives(
    pool_df: pl.DataFrame,
    df: pl.DataFrame,
    k: int,
    top_n: int = 10_000,
    seed: int = 42,
) -> pl.DataFrame:
    n = df.height * k

    top_items = (
        pool_df.group_by("item_id")
        .len()
        .sort("len", descending=True)
        .head(top_n)
        .select("item_id")
    )

    neg = pl.DataFrame(
        {
            "uid": pl.concat([df.get_column("uid")] * k, rechunk=True),
            "item_id": top_items.get_column("item_id").sample(
                n=n, with_replacement=True, seed=seed
            ),
            "target": pl.repeat(0, n, eager=True),
        }
    )

    pos = df.select(["uid", "item_id"]).with_columns(pl.lit(1).alias("target"))
    return pl.concat([pos, neg], how="vertical")


In [9]:
def add_item_popularity(train: pl.DataFrame, df: pl.DataFrame) -> pl.DataFrame:
    pop = train.group_by("item_id").len().rename({"len": "item_popularity"})
    return df.join(pop, on="item_id", how="left").with_columns(
        pl.col("item_popularity").fill_null(0)
    )


def add_user_count_likes(train: pl.DataFrame, df: pl.DataFrame) -> pl.DataFrame:
    pop = train.group_by("uid").len().rename({"len": "user_count_likes"})
    return df.join(pop, on="uid", how="left").with_columns(
        pl.col("user_count_likes").fill_null(0)
    )


def add_item_organic_share(train: pl.DataFrame, df: pl.DataFrame) -> pl.DataFrame:
    share = train.group_by("item_id").agg(
        pl.col("is_organic").mean().alias("item_organic_share")
    )
    return df.join(share, on="item_id", how="left").with_columns(
        pl.col("item_organic_share").fill_null(0)
    )


def add_user_organic_share(train: pl.DataFrame, df: pl.DataFrame) -> pl.DataFrame:
    share = train.group_by("uid").agg(
        pl.col("is_organic").mean().alias("user_organic_share")
    )
    return df.join(share, on="uid", how="left").with_columns(
        pl.col("user_organic_share").fill_null(0)
    )


def add_user_artist_like_cnt(
    events: pl.DataFrame, df: pl.DataFrame, item2artist: pl.DataFrame
) -> pl.DataFrame:
    stats = (
        events.select(["uid", "item_id"])
        .join(item2artist, on="item_id", how="left")
        .group_by(["uid", "artist_id"])
        .len()
        .rename({"len": "user_artist_like_cnt"})
    )

    return (
        df.join(item2artist, on="item_id", how="left")
        .join(stats, on=["uid", "artist_id"], how="left")
        .with_columns(pl.col("user_artist_like_cnt").fill_null(0))
        .drop("artist_id")
    )

In [10]:
def make_features(data_part1, data_part2, artists, add_negatives: bool = True):
    if add_negatives:
        train = add_popular_random_negatives(data_part1, data_part2, 10)
    else:
        train = data_part2.with_columns(pl.lit(1).alias("target"))

    train = add_item_popularity(data_part1, train)
    train = add_user_count_likes(data_part1, train)
    train = add_item_organic_share(data_part1, train)
    train = add_user_organic_share(data_part1, train)
    train = add_user_artist_like_cnt(data_part1, train, artists)
    return train

In [11]:
train = make_features(data_part1, data_part2, artists)
val = make_features(data_part2, data_part3, artists)


#### Обучаем катбуст:

In [12]:
METRICS = [
    "AUC",
    "CrossEntropy",
    "Precision",
    "Recall",
    "Accuracy",
    "LogLikelihoodOfPrediction",
]

train_pool = Pool(
    data=train.drop(["target", "item_id", "uid"]),
    label=train["target"],
)

val_pool = Pool(
    data=val.drop(["target", "item_id", "uid"]),
    label=val["target"],
)

model = CatBoostClassifier(
    iterations=100,
    learning_rate=0.1,
    depth=4,
    l2_leaf_reg=10,
    loss_function="Logloss",
    eval_metric="AUC",
    custom_metric=METRICS,
    early_stopping_rounds=100,
    verbose=10,
    use_best_model=True,
    task_type="GPU",
)

model.fit(train_pool, eval_set=val_pool, plot=True)

imps = model.get_feature_importance(type="PredictionValuesChange")
pairs = sorted(zip(model.feature_names_, imps), key=lambda x: x[1], reverse=True)

for name, val in pairs:
    print(f"{name}: {val}")


MetricVisualizer(layout=Layout(align_self='stretch', height='500px'))

Default metric period is 5 because AUC, LogLikelihoodOfPrediction is/are not implemented for GPU
Metric LogLikelihoodOfPrediction is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


0:	test: 0.7954367	best: 0.7954367 (0)	total: 29.9ms	remaining: 2.96s
10:	test: 0.8428430	best: 0.8441557 (9)	total: 113ms	remaining: 914ms
20:	test: 0.8463316	best: 0.8463316 (20)	total: 194ms	remaining: 730ms
30:	test: 0.8484659	best: 0.8484659 (30)	total: 272ms	remaining: 605ms
40:	test: 0.8502936	best: 0.8502936 (40)	total: 354ms	remaining: 509ms
50:	test: 0.8505371	best: 0.8505814 (45)	total: 432ms	remaining: 415ms
60:	test: 0.8506726	best: 0.8507376 (59)	total: 511ms	remaining: 326ms
70:	test: 0.8509392	best: 0.8509392 (70)	total: 592ms	remaining: 242ms
80:	test: 0.8509069	best: 0.8509442 (71)	total: 670ms	remaining: 157ms
90:	test: 0.8509075	best: 0.8509442 (71)	total: 749ms	remaining: 74.1ms
99:	test: 0.8509154	best: 0.8509478 (96)	total: 821ms	remaining: 0us
bestTest = 0.8509477973
bestIteration = 96
Shrink model to first 97 iterations.
item_popularity: 98.4894267354645
user_count_likes: 0.9341269386333617
item_organic_share: 0.30626115914059515
user_artist_like_cnt: 0.1873983

### 🔍  Retrieval:

#### Кандидатогенератор популярных треков

In [33]:
popular_tracks = (
    data_part2.group_by("item_id")
    .agg(pl.len())
    .sort("len", descending=True)[:100]
    .select(["item_id"])
)


In [34]:
positive_interactions = data_part3.select(["uid", "item_id"])
val_users = data_part3.select(["uid"]).unique()

test = val_users.select(["uid"]).join(popular_tracks, how="cross")
test = make_features(data_part2, test, artists, add_negatives=False)


#### Применяем модель

In [35]:
from utils.utils import recall_at_k

test_pool = Pool(
    data=test.drop(["item_id", "uid"]),
)

scores = model.predict_proba(test_pool)[:, 1]
scores_pl = (
    test.select(["uid", "item_id"])
    .with_columns(pl.Series("score", scores))
    .sort(["uid", "score"], descending=[False, True])
    .with_columns(
        [
            pl.col("score")
            .rank(method="ordinal", descending=True)
            .over("uid")
            .alias("rank")
        ]
    )
    .sort(["uid", "rank"])
)

for k in [30, 100, 300, 1000]:
    recall_k = recall_at_k(
        positive_interactions=positive_interactions,
        candidates=scores_pl,
        k=k,
    )
    print(f"Recall@{k}: {recall_k}")

Recall@30: 0.025064776476262304
Recall@100: 0.04755157746753479
Recall@300: 0.04755157746753479
Recall@1000: 0.04755157746753479


In [ ]:
# submit = (
#     test.select(["uid", "item_id"])
#     .with_columns(pl.Series("pred", pred))
#     .sort(["uid", "pred"], descending=[False, True])
#     .group_by("uid")
#     .agg(pl.col("item_id").head(100).cast(pl.Utf8).str.join(" ").alias("item_ids"))
# )

# submit

uid,item_ids
i64,str
89,"""7599492 3971215 4899017 736814…"
153,"""7599492 3971215 4899017 647068…"
164,"""7599492 3971215 6470688 736814…"
216,"""3971215 7599492 7368148 489901…"
291,"""3971215 7599492 6470688 489901…"
…,…
999735,"""7599492 3971215 7368148 647068…"
999737,"""3971215 7599492 7368148 489901…"
999779,"""7599492 3971215 6470688 489901…"


In [ ]:
submit.write_csv("catboost.csv")  # скор ~ 0.043

### Что дальше?

- Правильная оффлайн валидация (За какие даты собран тест?)
- Правильно собранный пул для обучения
- Больше фичей (Как сделать фичи из эмбеддингов?)
- Более богатые негативы
- Более богатые кандидатогенераторы
- CatBoostClassifier?
- Гиперпараметры модели
- Больше данных
- Учиться на всех данных
<center><img src="Timesplit2.svg" width="1100" /></center>